In [1]:
# config
samples = ["A1", "A2", "B2", "C2", "D1"]
input_path = "/scratch/leuven/357/vsc35768/spatial-transcriptomics/raw_data"
cellpose_model_path = "/scratch/leuven/357/vsc35768/.cache/cellpose/cpsam"
plotting = False
on_hpc = True
unite_testing = False

In [2]:
from pathlib import Path
import pandas as pd
import os
import gc
from datetime import date

from dask_image import imread
from spatialdata import SpatialData

from spatialdata.transformations import set_transformation, Identity
from spatialdata.models import PointsModel

import harpy as hp

from packaging import version

/data/leuven/357/vsc35768/miniconda3/envs/st-analysis/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/data/leuven/357/vsc35768/miniconda3/envs/st-analysis/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [3]:
if on_hpc:
    # setting cache to scratch for torch and cellpose
    os.environ['TORCH_HOME'] = os.path.join(os.environ['VSC_SCRATCH'], '.cache/torch')
    os.environ['CELLPOSE_LOCAL_MODELS_PATH'] = os.path.join(os.environ['VSC_SCRATCH'], '.cache/cellpose')

    print(f"TORCH_HOME: {os.environ.get('TORCH_HOME')}")
    print(f"CELLPOSE_LOCAL_MODELS_PATH: {os.environ.get('CELLPOSE_LOCAL_MODELS_PATH')}")

    # Check GPU availability in PyTorch
    import torch
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    pass

TORCH_HOME: /scratch/leuven/357/vsc35768/.cache/torch
CELLPOSE_LOCAL_MODELS_PATH: /scratch/leuven/357/vsc35768/.cache/cellpose
CUDA available: True
GPU count: 1
GPU name: NVIDIA A100-SXM4-80GB
GPU memory: 85.09 GB


In [4]:
import cellpose
import torch
from harpy.image import cellpose_callable

## Loading the data

In [5]:
def load_images_transcritps(folder: str, samples: list[str]) -> SpatialData:
    sdata = SpatialData()
    folder = Path(folder)
    for sample in samples:
        # load in the images
        matches = sorted(folder.glob(f"{sample}_DAPI.tiff"))
        if matches:
            p = matches[0]
            sdata = hp.im.add_image_layer(
                sdata,
                arr = imread.imread(str(p)),
                output_layer = f"{sample}_DAPI",
                transformations = {sample: Identity()},
                overwrite=True,
            )
        else:
            print(f"Warning: No DAPI image found for {sample}")
        

        # loading the transcripts
        df = pd.read_csv(
            Path(folder) / f"{sample}_results.txt",
            sep = r"\s+",
            header = None,
            names = ["x", "y", "z", "gene"],
            engine = "python",
        )
        points = PointsModel.parse(df, coordinates={"x": "x", "y": "y"})
        # set coordinate system for the transcripts
        points.attrs['transform'] = {} # ensuring no global coordinate system is set
        set_transformation(points, transformation = Identity(), to_coordinate_system = sample)
        sdata.points[f"{sample}_transcripts"] = points

    return sdata

In [6]:
sdata = load_images_transcritps(
    folder = input_path,
    samples = samples
)
sdata

2026-01-07 16:01:13.918 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'A1_DAPI'


INFO     Column `z` in `data` will be ignored since the data is 2D.                                                


2026-01-07 16:01:27.397 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'A2_DAPI'


INFO     Column `z` in `data` will be ignored since the data is 2D.                                                


2026-01-07 16:01:35.695 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'B2_DAPI'


...

2026-01-07 16:01:44.473 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'C2_DAPI'


INFO     Column `z` in `data` will be ignored since the data is 2D.                                                


2026-01-07 16:01:58.326 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'D1_DAPI'


INFO     Column `z` in `data` will be ignored since the data is 2D.                                                


SpatialData object
├── Images
│     ├── 'A1_DAPI': DataArray[cyx] (1, 12864, 19296)
│     ├── 'A2_DAPI': DataArray[cyx] (1, 10720, 17152)
│     ├── 'B2_DAPI': DataArray[cyx] (1, 10720, 19296)
│     ├── 'C2_DAPI': DataArray[cyx] (1, 12864, 21440)
│     └── 'D1_DAPI': DataArray[cyx] (1, 12864, 19296)
└── Points
      ├── 'A1_transcripts': DataFrame with shape: (<Delayed>, 3) (2D points)
      ├── 'A2_transcripts': DataFrame with shape: (<Delayed>, 3) (2D points)
      ├── 'B2_transcripts': DataFrame with shape: (<Delayed>, 3) (2D points)
      ├── 'C2_transcripts': DataFrame with shape: (<Delayed>, 3) (2D points)
      └── 'D1_transcripts': DataFrame with shape: (<Delayed>, 3) (2D points)
with coordinate systems:
    ▸ 'A1', with elements:
        A1_DAPI (Images), A1_transcripts (Points)
    ▸ 'A2', with elements:
        A2_DAPI (Images), A2_transcripts (Points)
    ▸ 'B2', with elements:
        B2_DAPI (Images), B2_transcripts (Points)
    ▸ 'C2', with elements:
        C2_DAPI (Imag

In [ ]:
sdata.images["A1_DAPI"]

## Image processing

In [7]:
# min max filtering
for sample in samples:
    sdata = hp.im.min_max_filtering(
        sdata,
        img_layer = f"{sample}_DAPI",                 # e.g. "A1_DAPI"
        output_layer = f"{sample}_min_max_filtered",  # e.g. "A1_min_max_filtered"
        size_min_max_filter = 51,
        overwrite = True,
    )
if plotting == True:
    for sample in samples:
        hp.pl.plot_image(
            sdata, 
            img_layer = [f"{sample}_DAPI", f"{sample}_min_max_filtered"], 
            crd = [4000, 8000, 6000, 8000], 
            figsize = (20,20),
            to_coordinate_system = sample
        )

2026-01-07 16:03:01.594 | INFO     | harpy.image._map:_precondition:339 - 'combine_z' is False, but not all 'z-slices' spefified in 'fn_kwargs'/'func' ({'size_min_max_filter': 51}/<function min_max_filtering.<locals>._apply_min_max_filter at 0x14a8bf5e7600>). Specifying z-slices ([0]).
2026-01-07 16:03:01.595 | INFO     | harpy.image._map:_precondition:345 - 'combine_c' is False, but not all channels spefified in 'fn_kwargs'/'func' ({np.int64(0): {'size_min_max_filter': 51}}/{np.int64(0): <function min_max_filtering.<locals>._apply_min_max_filter at 0x14a8bf5e7600>}). Specifying channels ([0]).
2026-01-07 16:03:11.285 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'A1_min_max_filtered'
2026-01-07 16:03:11.287 | INFO     | harpy.image._map:_precondition:339 - 'combine_z' is False, but not all 'z-slices' spefified in 'fn_kwargs'/'func' ({'size_min_max_filter': 51}/<function min_max_filtering.<locals>._apply_min_max_filter at 0x14a905bfe5c0>). Specifying z-slice

In [8]:
# enhance contrast
for sample in samples:
    sdata = hp.im.enhance_contrast(
        sdata,
        img_layer = f"{sample}_min_max_filtered",
        output_layer = f"{sample}_clahe",
        contrast_clip = 20,
        chunks = 20000,
        overwrite = True
    )

# Plot the contrast enhanced image
if plotting == True:
    for sample in samples:
        hp.pl.plot_image(
            sdata, 
            img_layer = [f"{sample}_min_max_filtered", f"{sample}_clahe"], 
            crd = [4000, 8000, 6000, 8000], 
            figsize = (20,20),
            to_coordinate_system = sample
        )

2026-01-07 16:03:53.622 | INFO     | harpy.image._map:_precondition:339 - 'combine_z' is False, but not all 'z-slices' spefified in 'fn_kwargs'/'func' ({'contrast_clip': 20}/<function enhance_contrast.<locals>._apply_clahe at 0x14a905bf7920>). Specifying z-slices ([0]).
2026-01-07 16:03:53.623 | INFO     | harpy.image._map:_precondition:345 - 'combine_c' is False, but not all channels spefified in 'fn_kwargs'/'func' ({np.int64(0): {'contrast_clip': 20}}/{np.int64(0): <function enhance_contrast.<locals>._apply_clahe at 0x14a905bf7920>}). Specifying channels ([0]).
2026-01-07 16:03:54.405 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'A1_clahe'
2026-01-07 16:03:54.408 | INFO     | harpy.image._map:_precondition:339 - 'combine_z' is False, but not all 'z-slices' spefified in 'fn_kwargs'/'func' ({'contrast_clip': 20}/<function enhance_contrast.<locals>._apply_clahe at 0x14a905bf7920>). Specifying z-slices ([0]).
2026-01-07 16:03:54.409 | INFO     | harpy.image._

In [9]:
sdata

SpatialData object
├── Images
│     ├── 'A1_DAPI': DataArray[cyx] (1, 12864, 19296)
│     ├── 'A1_clahe': DataArray[cyx] (1, 12864, 19296)
│     ├── 'A1_min_max_filtered': DataArray[cyx] (1, 12864, 19296)
│     ├── 'A2_DAPI': DataArray[cyx] (1, 10720, 17152)
│     ├── 'A2_clahe': DataArray[cyx] (1, 10720, 17152)
│     ├── 'A2_min_max_filtered': DataArray[cyx] (1, 10720, 17152)
│     ├── 'B2_DAPI': DataArray[cyx] (1, 10720, 19296)
│     ├── 'B2_clahe': DataArray[cyx] (1, 10720, 19296)
│     ├── 'B2_min_max_filtered': DataArray[cyx] (1, 10720, 19296)
│     ├── 'C2_DAPI': DataArray[cyx] (1, 12864, 21440)
│     ├── 'C2_clahe': DataArray[cyx] (1, 12864, 21440)
│     ├── 'C2_min_max_filtered': DataArray[cyx] (1, 12864, 21440)
│     ├── 'D1_DAPI': DataArray[cyx] (1, 12864, 19296)
│     ├── 'D1_clahe': DataArray[cyx] (1, 12864, 19296)
│     └── 'D1_min_max_filtered': DataArray[cyx] (1, 12864, 19296)
└── Points
      ├── 'A1_transcripts': DataFrame with shape: (<Delayed>, 3) (2D points)
      ├

## Cell segmentation

In [10]:
# checking what is available on the system
cellpose_version = version.parse(cellpose.version)
if torch.backends.mps.is_available() and cellpose_version >= version.parse("4.0"):  # mps bugged in cellpose < 4.0
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Using device: {device}.")

Using device: cuda.


In [ ]:
# from dask.distributed import Client, LocalCluster

# # # Create a local Dask cluster
# cluster = LocalCluster(
#      n_workers=4,              # Number of worker processes
#      threads_per_worker=2,    # Number of threads per worker
#      processes=False,
#      memory_limit="auto",      # Memory limit per worker
#  )

# # # Connect a Client to the cluster
# client = Client(cluster)

# # # Print the Dask dashboard link
# print(client.dashboard_link)

In [11]:
# this needs to be done on a GPU (took 8min on NVIDIA A100-SXM4-80GB)
for sample in samples:
    sdata = hp.im.segment(
        sdata,
        img_layer= f"{sample}_clahe", # The image layer in sdata to be segmented.
        chunks = 4096,
        depth = 40,
        model = cellpose_callable,
        # parameters that will be passed to the callable _cellpose:
        pretrained_model = cellpose_model_path, 
        device = device,
        diameter = 50,
        flow_threshold = 0.6,
        cellprob_threshold = -6,
        min_size = 40,
        output_labels_layer = f"{sample}_segmentation_mask",
        output_shapes_layer = f"{sample}_segmentation_mask_boundaries",
        #crd=[6000, 10096, 6000, 10096] if unit_testing else None, 
        to_coordinate_system = sample,
        overwrite = True,
    )
    gc.collect() # freeing memory
    torch.cuda.empty_cache() # empty caching

channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
channels deprecated

In [12]:
sdata

SpatialData object
├── Images
│     ├── 'A1_DAPI': DataArray[cyx] (1, 12864, 19296)
│     ├── 'A1_clahe': DataArray[cyx] (1, 12864, 19296)
│     ├── 'A1_min_max_filtered': DataArray[cyx] (1, 12864, 19296)
│     ├── 'A2_DAPI': DataArray[cyx] (1, 10720, 17152)
│     ├── 'A2_clahe': DataArray[cyx] (1, 10720, 17152)
│     ├── 'A2_min_max_filtered': DataArray[cyx] (1, 10720, 17152)
│     ├── 'B2_DAPI': DataArray[cyx] (1, 10720, 19296)
│     ├── 'B2_clahe': DataArray[cyx] (1, 10720, 19296)
│     ├── 'B2_min_max_filtered': DataArray[cyx] (1, 10720, 19296)
│     ├── 'C2_DAPI': DataArray[cyx] (1, 12864, 21440)
│     ├── 'C2_clahe': DataArray[cyx] (1, 12864, 21440)
│     ├── 'C2_min_max_filtered': DataArray[cyx] (1, 12864, 21440)
│     ├── 'D1_DAPI': DataArray[cyx] (1, 12864, 19296)
│     ├── 'D1_clahe': DataArray[cyx] (1, 12864, 19296)
│     └── 'D1_min_max_filtered': DataArray[cyx] (1, 12864, 19296)
├── Labels
│     ├── 'A1_segmentation_mask': DataArray[yx] (12864, 19296)
│     ├── 'A2_segment

In [13]:
if plotting == True:
    hp.pl.plot_shapes(
        sdata, 
        img_layer = "A1_clahe", 
        shapes_layer = "A1_segmentation_mask_boundaries", 
        figsize=(10,10), 
        to_coordinate_system = "A1",
        crd = [2000, 4000, 2000, 4000]
    )

In [15]:
# write the object to zarr
sdata.write(f"/scratch/leuven/357/vsc35768/spatial-transcriptomics/intermediate_results/20260107.zarr", overwrite=True)

: 